# Certilab Adaptive RAG — Demo en vivo

Este notebook ejecuta el grafo **Adaptive RAG** canónico con 7 nodos y 2 loops
de auto-corrección, usando LangGraph + OpenAI + Tavily.

**Referencia**: [Building an Adaptive RAG System](https://levelup.gitconnected.com/building-an-adaptive-rag-system-with-langgraph-openai-and-tavily-c4ee39d2f021)

### Requisitos
- `OPENAI_API_KEY` (Colab: Secrets 🔑, local: `.env`)
- `TAVILY_API_KEY` opcional

💡 **Colab**: ejecutá la celda 1 para clonar e instalar. **Local**: salteala.


In [ ]:
# === Solo para Colab: clonar repo e instalar dependencias ===
import os, sys
try:
    from google.colab import userdata
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

if _IN_COLAB:
    !git clone https://github.com/emersonheto/certilab-adaptive-rag.git /content/certilab-adaptive-rag
    %cd /content/certilab-adaptive-rag
    !pip install -q langgraph langchain-openai langchain-core openai tavily-python pydantic pydantic-settings python-dotenv
    print("✅ Repo clonado y dependencias instaladas")
else:
    print("ℹ️  Ejecutando en local — el repo ya está clonado")


Cloning into '/content/certilab-adaptive-rag'...
remote: Enumerating objects: 159, done.
remote: Counting objects: 100% (159/159), done.
remote: Compressing objects: 100% (111/111), done.
remote: Total 159 (delta 57), reused 144 (delta 42), pack-reused 0 (from 0)
Receiving objects: 100% (159/159), 3.04 MiB | 16.36 MiB/s, done.
Resolving deltas: 100% (57/57), done.
/content/certilab-adaptive-rag
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.3/26.3 MB 67.0 MB/s eta 0:00:00:00:0100:01


In [ ]:
# === Configurar API keys ===
if _IN_COLAB:
    try:
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except Exception:
        # Si falla Colab Secrets, pedir la key manualmente
        key = input("🔑 Ingresá tu OPENAI_API_KEY (sk-...): ")
        os.environ["OPENAI_API_KEY"] = key.strip()
    try:
        os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
    except Exception:
        pass
    print("✅ Keys configuradas")
else:
    from dotenv import load_dotenv
    from pathlib import Path
    _env = Path.cwd() / ".env"
    load_dotenv(_env if _env.exists() else None, override=False)

if not os.getenv("OPENAI_API_KEY"):
    print("❌ OPENAI_API_KEY no encontrada")
    if _IN_COLAB:
        print("   Agregala en Secrets (🔑) o ingresala cuando te la pida")
    else:
        print("   Creá .env con OPENAI_API_KEY=sk-... en la raíz del proyecto")
    sys.exit(1)
print(f"✅ OPENAI_API_KEY configurada | TAVILY: {'✅' if os.getenv('TAVILY_API_KEY') else '❌'}")


In [ ]:
from app.adaptive_rag.graph import build_graph
from app.adaptive_rag.state import AdaptiveRAGState
from app.config import Settings
from app.domain.models import Role
from app.security.access_control import Principal, scope_from_principal
from app.tools.web_search import TavilyWebSearch, WebSearchConfig
print("✅ Imports OK")


In [ ]:
from pathlib import Path
settings = Settings()
print(f"Modo: {settings.app_mode}")

if settings.app_mode != "real":
    # Mock mode — datos en memoria
    from app.ingestion.loader import load_certificates, load_customers, load_histories, load_pdf_texts
    from app.ingestion.splitter import build_pdf_chunks
    from app.ingestion.indexer import InMemoryVectorIndex
    data_dir = Path("data")
    certificates = load_certificates(data_dir)
    load_customers(data_dir); load_histories(data_dir)
    chunks = build_pdf_chunks(certificates, load_pdf_texts(data_dir))
    index = InMemoryVectorIndex(chunks=chunks)
    embedding_provider = None
    print(f"✅ Mock: {len(certificates)} certificados en memoria")
else:
    from app.retrieval.qdrant_index import QdrantVectorIndex
    from app.tools.embeddings import EmbeddingProviderConfig, EmbeddingsProvider
    embedding_provider = EmbeddingsProvider(EmbeddingProviderConfig.from_settings(settings))
    index = QdrantVectorIndex.from_settings(settings, embedding_provider)
    print(f"✅ Qdrant conectado | {settings.openai_embedding_model}")

tavily_key = settings.tavily_api_key
web_search = TavilyWebSearch(WebSearchConfig(tavily_api_key=tavily_key))
graph = build_graph(index=index, embeddings=embedding_provider, web_search=web_search, settings=settings)
print(f"✅ Grafo compilado | Web search: {'✅' if tavily_key else '❌'}")


In [ ]:
def make_state(question: str) -> AdaptiveRAGState:
    p = Principal(role=Role.ADMIN, customer_id=None, user_id=1)
    s = scope_from_principal(p)
    return {"question": question, "generation": "", "documents": [],
            "web_results": [], "route": "", "rewrite_count": 0,
            "regenerate_count": 0, "hallucination_verdict": "",
            "answer_verdict": "", "principal": p, "scope": s}

def run(question: str):
    for step in graph.stream(make_state(question)):
        for name, out in step.items():
            print(f"--- {name} ---")
            if isinstance(out, dict):
                for k, v in out.items():
                    s = repr(v); print(f"  {k}: {s[:180]}")
            print()
print("✅ Helpers")


## 🔍 Query A: Vectorstore — Procedimiento INDECOPI


In [ ]:
run("¿Qué procedimiento de calibración sigue la norma INDECOPI?")


## 🎯 Query B: Tenant isolation — ALS PERU


In [ ]:
run("¿Qué certificados tiene ALS PERU?")


## 🔄 Self-Correction Loops
1. **Rewrite loop**: docs irrelevantes → reescribe → reintenta (máx 3)
2. **Regenerate loop**: alucina → regenera (máx 2). No útil → reescribe


### 🔄 Loop 1: Reescritura


In [ ]:
run("Dame info de calibración")


### 📊 Query D: Datos de tablas — Medición 105°C


In [ ]:
run("¿Cuál fue la temperatura máxima registrada a 105°C?")


### 🏷️ Query E: Metadatos — Fecha + Tipo


In [ ]:
run("¿Qué certificados acreditados se emitieron en mayo 2026?")


## 📊 Visualización del grafo


In [ ]:
from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))


## ✅ Resumen
- ✅ Enrutamiento adaptativo
- ✅ Tenant isolation
- ✅ Gradeo de relevancia
- ✅ Reescritura de query (máx 3)
- ✅ Verificación de alucinaciones (máx 2)
- ✅ Datos reales: 154 certificados
- ✅ Pipeline: PyMuPDF + Camelot + Unstructured → Qdrant

**Stack**: Python 3.11 | LangGraph | OpenAI | Tavily | Pydantic v2 | Qdrant
